In [11]:
# Feature engineering
#Setup
from pathlib import Path

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"

# print(ROOT)

In [ ]:
# Import cleaned CSV files
import pandas as pd

# print(pd.read_csv(DATA / "cleaned" / "price_cleaned.csv", nrows=5))
# print(pd.read_csv(DATA / "cleaned" / "price_cleaned.csv", nrows=0).columns.tolist())


price = pd.read_csv(
    DATA / "cleaned" / "price_cleaned.csv",
    parse_dates=["date"],
    index_col="date"
)

onchain = pd.read_csv(
    DATA / "cleaned" / "onchain_cleaned.csv",
    parse_dates=["date"],
    index_col="date"
)

                        date     open     high      low    close
0  2017-11-13 00:00:00+00:00  5839.94  6697.47  5699.99  6465.99
1  2017-11-14 00:00:00+00:00  6465.99  6684.98  6311.07  6574.99
2  2017-11-15 00:00:00+00:00  6575.99  7298.00  6575.99  7240.06
3  2017-11-16 00:00:00+00:00  7240.14  7940.00  7076.00  7864.50
4  2017-11-17 00:00:00+00:00  7876.98  7989.00  7451.00  7699.19
['date', 'open', 'high', 'low', 'close']


In [13]:
# 1, 7, and 30 day returns
### R_t=(P_t-P_(t-1))/P_(t-1) 
###     R_t = return on current day
###     P_t= Closing price on current day
###     P_(t-1)= Closing price on previous day
### Provides daily percentage change

price["return_1d"] = price["close"].pct_change(fill_method=None)

price["return_7d"] = price["close"].pct_change(periods=7, fill_method=None)

price["return_30d"] = price["close"].pct_change(periods=30, fill_method=None)


In [14]:
# Running averages
# 7 & 30 day moving averages

price["ma_7d"] = price["close"].rolling(7).mean()
price["ma_30d"] = price["close"].rolling(30).mean()


In [15]:
# Rolling volatility -- measures magnitude of recent plrice fluctuation using std. dev.
price["volatility_7d"] = (price["return_1d"].rolling(7).std())

price["volatility_30d"] = (price["return_1d"].rolling(30).std())


In [16]:
# Previous days returns -- lagged returns
price["return_lag_1"] = price["return_1d"].shift(1)
price["return_lag_2"] = price["return_1d"].shift(2)

price["price_range"] = ((price["high"] - price["low"]) / price["open"])

In [17]:
# Onchain feature engineering
import numpy as np

metrics = [
    "active_addresses",
    "transaction_count",
    "hash_rate",
    "transaction_fees"
]

for metric in metrics:
    onchain[f"{metric}_change_7d"] = (
        onchain[metric].pct_change(
            periods=7,
            fill_method=None
        )
    )

# Replace infinite values w/missing values
onchain = onchain.replace(
    [np.inf, -np.inf],
    np.nan
)

In [18]:
# Account for on-chain reportign delays to reduce look-ahead bias. 
# On-chain metrics available on day t are used to predict close
# price direction on day t + 1
# On-chain features are shifted

onchain = onchain.shift(1)

In [20]:
# Create target variable (binary classification)
next_close = price["close"].shift(-1)

price["target"] = (
    next_close > price["close"]
).astype("Int64")

# Final observation excluded if following day's closing price is unknown
price.loc[
    next_close.isna() | price["close"].isna(),
    "target"
] = pd.NA

# Retain next-day return for trading strategy evaluation
price["next_day_return"] = (next_close / price["close"] - 1)



In [ ]:
# Merge datasets

from math import comb


combined = price.join(
    onchain,
    how="left",
    validate="one_to_one"
)

# Inspect resulting datasets
print(combined.head())
print(combined.shape)
print(combined.isna().sum())

# Restrict dataset to common modeling period
combined = combined.loc[
    "2018-01-01":"2026-05-23"
].copy()

print(combined.isna().sum())

In [ ]:
# Verify
print(combined.head())
print(combined.tail())
print(combined.shape)
print(combined.index.min())
print(combined.index.max())
print(combined.isna().sum())